# Training Pipeline - ChatKasir

- Nama: Achmad Rif'an (AI-1 Model Architect)
- Minggu: 2 - Pengembangan Fitur Inti (27 April - 1 Mei)


## 1. Setup Google Colab dengan GPU

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# verifikasi versi
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

TensorFlow : 2.20.0
NumPy      : 2.0.2
Pandas     : 2.2.2


In [2]:
# verifikasi GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU tersedia: {len(gpus) > 0}")

if gpus:
    # tampilkan detail GPU yang aktif agar terdokumentasi
    for gpu in gpus:
        print(f"Nama GPU    : {gpu.name}")

    # aktifkan memory growth - GPU tidak langsung mengambil semua VRAM
    # tapi mengalokasikan secara bertahap sesuai kebutuhan
    # ini mencegah crash "out of memory" di awal training
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth: aktif")
else:
    print("PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type")


GPU tersedia: True
Nama GPU    : /physical_device:GPU:0
Memory growth: aktif


## 2. Load Dataset

In [3]:
# load dataset food
url_food_utama = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"

df_food = pd.read_csv(url_food_utama)
df_food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18558 entries, 0 to 18557
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    18558 non-null  object
dtypes: object(1)
memory usage: 145.1+ KB


In [4]:
# load dataset slang
url_slang_utama = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"

df_slang = pd.read_csv(url_slang_utama)
df_slang.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   slang   1231 non-null   object
 1   formal  1231 non-null   object
dtypes: object(2)
memory usage: 19.4+ KB


In [5]:
# load dataset sintetis
url_synthetic_10000 = "https://drive.google.com/uc?id=15luIrYGJEZpbH-Wf7foXHXo3LBt0MBqU"

df_synthetic = pd.read_csv(url_synthetic_10000)
df_synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   input_text    10050 non-null  object
 1   product       10050 non-null  object
 2   quantity      10050 non-null  int64 
 3   price_satuan  10050 non-null  int64 
 4   pattern       10050 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 392.7+ KB


## 3. Pipeline Data Loading
Ada 6 tahap yang dilakukan:

1. Memuat dan memvalidasi dataset
2. Tokenisasi: mengubah teks percakapan menjadi array integer
3. Encoding label: setiap nama produk dipetakan ke integer
4. Normalisasi price dan persiapan array
5. Split dataset menjadi training set dan validation set
6. Membangun tf.data.Dataset untuk mengirimkan data ke model saat training


### Tahap 1: Validasi Dataset Sintetis

In [6]:
print(f"Shape dataset   : {df_synthetic.shape}")
print(f"Kolom yang ada  : {list(df_synthetic.columns)}")


Shape dataset   : (10050, 5)
Kolom yang ada  : ['input_text', 'product', 'quantity', 'price_satuan', 'pattern']


In [7]:
# validasi nama kolom sesuai kesepakatan
kolom_wajib = ["input_text", "product", "quantity", "price_satuan", "pattern"]

kolom_hilang = [k for k in kolom_wajib if k not in df_synthetic.columns]
if kolom_hilang:
    print(f"ERROR: Kolom berikut tidak ditemukan: {kolom_hilang}")
    print("Minta Faradi (DS-1) untuk menyesuaikan nama kolom!")
else:
    print("Semua kolom sesuai")

Semua kolom sesuai


In [8]:
# tampilkan sampel data untuk inspeksi visual
print(f"Sampel 3 baris pertama:")
print(df_synthetic[["input_text", "product", "quantity", "price_satuan"]].head(3).to_string())

Sampel 3 baris pertama:
                                                                                                                  input_text                        product  quantity  price_satuan
0  minta 7 nasi kari limasari nasi putih ya [SEP] oke kak nasi kari limasari nasi putih harganya 22 ribu totalnya rp 154.000  nasi kari limasari nasi putih         7         22000
1                         bu mau pesen 4 mie goreng djawa [SEP] noted kak mie goreng djawa rp 48.000 per porsi totalnya 192k               mie goreng djawa         4         48000
2                                                                         4 teh obenk dong [SEP] oke kak pesanannya masuk ya                      teh obenk         4            -1


In [9]:
# cek distribusi pattern, memastikan proporsi seimbang
print(f"Distribusi pattern:")

for pattern, count in df_synthetic['pattern'].value_counts().sort_index().items():
    pct = count / len(df_synthetic) * 100
    print(f"Pola {pattern}: {count} baris ({pct:.1f}%)")

Distribusi pattern:
Pola 1: 3922 baris (39.0%)
Pola 2: 3888 baris (38.7%)
Pola 3: 2240 baris (22.3%)


In [10]:
# cek baris dengan harga satuan -1
n_null = (df_synthetic['price_satuan'] == -1).sum()

print(f"Baris tanpa harga satuan (-1): {n_null} ({n_null/len(df_synthetic):.1%})")

Baris tanpa harga satuan (-1): 1750 (17.4%)


In [11]:
# cek apakah ada nilai yang null (NaN)
print(f"Nilai NaN per kolom:")
print(df_synthetic[kolom_wajib].isnull().sum().to_string())

Nilai NaN per kolom:
input_text      0
product         0
quantity        0
price_satuan    0
pattern         0


### Tahap 2: Tokenisasi Teks
Mengubah teks chat obrolan pesanan pembeli dan penjual menjadi array integer menggunakan TextVectorization

In [12]:
# hyperparameter
# diambil dari 01_model_architecture.ipynb Minggu 1
VOCAB_SIZE = 10000
MAX_SEQ_LEN = 30

# TextVectorization
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE, # maksimum kata unik dalam kamus
    output_sequence_length=MAX_SEQ_LEN, # panjang urutan maksmimum
    output_mode="int", # output berupa integer (bukan one-hot)
    standardize="lower_and_strip_punctuation", # lowercase dan hapus tanda baca
    name="text_vectorizer"
)

# fitting vectorizer, membangun kamus kata dari seluruh input_text
# dilakukan sekali pada training set, bukan test set
# untuk menghindari data leakage dari test set ke training
print("Membangun kamus kata dari dataset...")
vectorizer.adapt(df_synthetic["input_text"].values)

Membangun kamus kata dari dataset...


In [13]:
# verifikasi vocabulary yang terbentuk
vocab = vectorizer.get_vocabulary()

print(f"Kamus terbentuk dengan {len(vocab)} kata unik")
print(f"10 kata pertama (termasuk token khusus): {vocab[:5]}")
print(f"10 kata paling umum: {vocab[2:12]}")

# catatan: vocab[0] = "" (padding), vocab[1] = "[UNK]" (unknown)
# kata-kata selanjutnya diurutkan dari yang paling sering muncul

Kamus terbentuk dengan 4056 kata unik
10 kata pertama (termasuk token khusus): ['', '[UNK]', np.str_('kak'), np.str_('sep'), np.str_('ya')]
10 kata paling umum: [np.str_('kak'), np.str_('sep'), np.str_('ya'), np.str_('totalnya'), np.str_('nasi'), np.str_('ayam'), np.str_('oke'), np.str_('rp'), np.str_('dong'), np.str_('siap')]


In [14]:
# uji coba tokenisasi pada 1 contoh teks
contoh_teks = df_synthetic["input_text"].iloc[1]
contoh_token = vectorizer([contoh_teks]).numpy()[0]

print(f"Contoh tokenisasi:")
print(f"Teks asli: '{contoh_teks}'")
print(f"Hasil token: {contoh_token}")

Contoh tokenisasi:
Teks asli: 'bu mau pesen 4 mie goreng djawa [SEP] noted kak mie goreng djawa rp 48.000 per porsi totalnya 192k'
Hasil token: [  66   13   27   38   24   17 1549    3   21    2   24   17 1549    9
  192   15   16    5 1603    0    0    0    0    0    0    0    0    0
    0    0]


### Tahap 3: Encoding Label Product
Mengubah nama produk dari teks menjadi integer

In [15]:
# StringLookup
# layer tensorflow yang memetakan string menjadi integer
# membuat kamus produk, tiap nama produk unik mendapat nomor urut yang konsisten
label_encoder_product = tf.keras.layers.StringLookup(
    output_mode="int",
    name="product_label_encoder"
)

# fit label_encoder_product dengan seluruh nama produk unik di dataset
label_encoder_product.adapt(df_synthetic["product"].values)

In [16]:
# jumlah kelas produk harus konsisten dengan NUM_PRODUCTS
# di arsitektur model 01_model_architecture.ipynb (+1 UNK)
NUM_PRODUCTS = label_encoder_product.vocabulary_size()

print(f"Jumlah kelas produk (NUM_PRODUCTS): {NUM_PRODUCTS}")
print(f"Catatan: angka ini harus sama dengan yang digunakan di build_model()")

Jumlah kelas produk (NUM_PRODUCTS): 4976
Catatan: angka ini harus sama dengan yang digunakan di build_model()


In [17]:
# uji coba encoding
contoh_produk = df_synthetic["product"].iloc[1]
contoh_encoded = label_encoder_product([contoh_produk]).numpy()[0]

print(f"\nContoh label encoding:")
print(f"  Nama produk : '{contoh_produk}'")
print(f"  Hasil encode: {contoh_encoded}")


Contoh label encoding:
  Nama produk : 'mie goreng djawa'
  Hasil encode: 912


In [18]:
# simpan vocabulary label encoder untuk digunakan saat inferensi
# dibutuhkan untuk mengubah integer kembali ke nama produk
# (de-encoding) saat model sudah terlatih
produk_vocab = label_encoder_product.get_vocabulary()

print(f"5 produk pertama dalam kamus: {produk_vocab[:5]}")

# produk_vocab[0] = "[UNK]" untuk produk yang tidak dikenal
# produk_vocab[1:] = nama-nama produk yang dikenal

5 produk pertama dalam kamus: ['[UNK]', np.str_('nasi telor kalimantan'), np.str_('es kopi tem'), np.str_('nasi nila saos tiram'), np.str_('ice kopi taro')]


### Tahap 4: Normalisasi Price dan Persiapan Array

In [19]:
# tokenisasi seluruh input_text sekaligus
X = vectorizer(df_synthetic["input_text"].values).numpy()

print(f"Shape X (input model): {X.shape}")

Shape X (input model): (10050, 30)


In [20]:
# label encoding untuk product
y_product = label_encoder_product(df_synthetic["product"].values).numpy()

print(f"Shape y_product: {y_product.shape}")

Shape y_product: (10050,)


In [21]:
# quantity langsung diambil sebagai integer
y_quantity = df_synthetic["quantity"].values.astype(np.float32)

print(f"Shape y_quantity: {y_quantity.shape}")
print(f"Range quantity: {y_quantity.min():.0f} - {y_quantity.max():.0f}")

Shape y_quantity: (10050,)
Range quantity: 1 - 10


In [22]:
# price: normalisasi dengan membagi 1000
# kecuali baris dengan sentinel -1 yang harus dibiarkan -1
y_price_raw = df_synthetic["price_satuan"].values.astype(np.float32)
y_price = np.where(
    y_price_raw == -1,       # kondisi: kalau nilainya sentinel -1
    -1.0,                    # jika benar: biarkan -1 (jangan dinormalisasi)
    y_price_raw / 1000.0     # jika salah: normalisasi dengan bagi 1000
)

print(f"Shape y_price             : {y_price.shape}")
print(f"Range price valid (÷1000) : {y_price[y_price != -1].min():.1f} – {y_price[y_price != -1].max():.1f}")
print(f"Jumlah sentinel -1        : {(y_price == -1).sum()}")

Shape y_price             : (10050,)
Range price valid (÷1000) : 3.0 – 75.0
Jumlah sentinel -1        : 1750


### Tahap 5: Split Train & Validation
Membagi dataset menjadi training set sebesar 80% dan validation set 20%

In [23]:
X_train, X_val, y_product_train, y_product_val, y_quantity_train, y_quantity_val, y_price_train, y_price_val = train_test_split(
    X, y_product, y_quantity, y_price,
    test_size=0.2,
    random_state=42, # memastikan split yang sama setiap kali dijalankan
    stratify=df_synthetic["pattern"].values # menjaga proporsi ketiga pola tetap seimbang
)

print(f"Training set   : {len(X_train)} baris ({len(X_train)/len(X):.0%})")
print(f"Validation set : {len(X_val)} baris ({len(X_val)/len(X):.0%})")

print(f"\nVerifikasi shape setelah split:")
print(f"X_train        : {X_train.shape}")
print(f"y_product_train: {y_product_train.shape}")
print(f"y_price_train  : {y_price_train.shape}")

Training set   : 8040 baris (80%)
Validation set : 2010 baris (20%)

Verifikasi shape setelah split:
X_train        : (8040, 30)
y_product_train: (8040,)
y_price_train  : (8040,)


### Tahap 6: Membangun tf.data.Dataset

Membungkus array numpy menjadi tf.data.Dataset yang siap digunakan untuk training atau validasi.

Parameter shuffle=True digunakan untuk training set agar urutan data acak setiap epoch. Mencegah model belajar pola berdasarkan urutan.

Untuk validation set, shuffle=False karena tidak diperlukan.

In [24]:
# model memproses 32 kalimat sekaligus dalam 1 langkah, bukan satu per satu
BATCH_SIZE = 32

def buat_dataset(X, y_product, y_quantity, y_price, shuffle=False):

    # dataset input: dictionary karena model.fit() mengharapkan format ini
    # saat model memiliki multiple inputs (meskipun di sini hanya satu input)
    dataset = tf.data.Dataset.from_tensor_slices((
        {"input_tokens": X}, # input ke model
        {   # label (target) untuk setiap output head
            "product": y_product,
            "quantity": y_quantity,
            "price_satuan": y_price
        }
    ))

    if shuffle:
        # buffer size = jumlah data
        dataset = dataset.shuffle(buffer_size=len(X), seed=42)

    dataset = dataset.batch(BATCH_SIZE)

    # prefetch(tf.data.AUTOTUNE): saat GPU sedang melatih batch ke-N,
    # CPU sudah mempersiapkan batch ke-N+1 di background
    # ini menghilangkan "waktu menunggu" antara batch dan mempercepat training
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_dataset = buat_dataset(
    X_train, y_product_train, y_quantity_train, y_price_train, shuffle=True
)

val_dataset = buat_dataset(
    X_val, y_product_val, y_quantity_val, y_price_val, shuffle=False
)

# hitung jumlah batch
n_train_batches = len(X_train) // BATCH_SIZE
n_val_batches   = len(X_val) // BATCH_SIZE

print(f"Training dataset   : {n_train_batches} batch x {BATCH_SIZE} sampel")
print(f"Validation dataset : {n_val_batches} batch x {BATCH_SIZE} sampel")

Training dataset   : 251 batch x 32 sampel
Validation dataset : 62 batch x 32 sampel


In [25]:
# verifikasi satu batch untuk memastikan format sudah benar
print(f"Verifikasi format satu batch:")

for inputs_batch, labels_batch in train_dataset.take(1):
    print(f"Input 'input_tokens' shape : {inputs_batch['input_tokens'].shape}")
    print(f"Label 'product' shape      : {labels_batch['product'].shape}")
    print(f"Label 'quantity' shape     : {labels_batch['quantity'].shape}")
    print(f"Label 'price_satuan' shape : {labels_batch['price_satuan'].shape}")

Verifikasi format satu batch:
Input 'input_tokens' shape : (32, 30)
Label 'product' shape      : (32,)
Label 'quantity' shape     : (32,)
Label 'price_satuan' shape : (32,)


## 4. Tokenisasi, Padding & Encoding Label
Bagian ini memperdalam dan memverifikasi secara eksplisit setiap tahap encoding yang sudah dijalankan di Bagian 3.

Tujuannya adalah memastikan setiap komponen pipeline bekerja benar sebelum digunakan untuk training model, dan mendokumentasikan keputusan teknis untuk referensi AI-2 (Denny) dan laporan teknis DS-2 (Salman).

### 4.1 Verifikasi Mendalam Tokenisasi (input_text)

In [26]:
# Tahap 2 di bagian sebelumnya sudah menjalankan tokenisasi secara massal.
# di sini verifikasi cara kerjanya secara eksplisit pada satu contoh
# untuk memastikan tidak ada kesalahan yang tersembunyi.

# ambil vocabulary yang sudah terbentuk dari vectorizer
vocab = vectorizer.get_vocabulary()
print(f"Total kata dalam kamus  : {len(vocab)}")
print(f"\nStruktur kamus (4 slot pertama):")
print(f"  vocab[0] = '{vocab[0]}' token PADDING (integer 0 = 'kosong')")
print(f"  vocab[1] = '{vocab[1]}' token [UNK] untuk kata tidak dikenal")
print(f"  vocab[2] = '{vocab[2]}' kata PALING SERING muncul di dataset")
print(f"  vocab[3] = '{vocab[3]}' kata terbanyak kedua")
print(f"  vocab[4:9] = {vocab[4:9]}")

Total kata dalam kamus  : 4056

Struktur kamus (4 slot pertama):
  vocab[0] = '' token PADDING (integer 0 = 'kosong')
  vocab[1] = '[UNK]' token [UNK] untuk kata tidak dikenal
  vocab[2] = 'kak' kata PALING SERING muncul di dataset
  vocab[3] = 'sep' kata terbanyak kedua
  vocab[4:9] = [np.str_('ya'), np.str_('totalnya'), np.str_('nasi'), np.str_('ayam'), np.str_('oke')]


In [27]:
# ambil satu contoh nyata dari dataset
idx_contoh    = 0
teks_contoh   = df_synthetic["input_text"].iloc[idx_contoh]
produk_contoh = df_synthetic["product"].iloc[idx_contoh]
qty_contoh    = df_synthetic["quantity"].iloc[idx_contoh]
price_contoh  = df_synthetic["price_satuan"].iloc[idx_contoh]

print(f"\nContoh baris ke-{idx_contoh}:")
print(f"input_text   : '{teks_contoh}'")
print(f"product      : '{produk_contoh}'")
print(f"quantity     : {qty_contoh}")
print(f"price_satuan : {price_contoh}")


Contoh baris ke-0:
input_text   : 'minta 7 nasi kari limasari nasi putih ya [SEP] oke kak nasi kari limasari nasi putih harganya 22 ribu totalnya rp 154.000'
product      : 'nasi kari limasari nasi putih'
quantity     : 7
price_satuan : 22000


In [28]:
# tokenisasi satu kalimat untuk inspeksi
token_result = vectorizer([teks_contoh]).numpy()[0]

print(f"Hasil tokenisasi (array {MAX_SEQ_LEN} integer):")
print(f"{token_result}")

Hasil tokenisasi (array 30 integer):
[  30   31    6  305  387    6  141    4    3    8    2    6  305  387
    6  141   28  667   14    5    9 3025    0    0    0    0    0    0
    0    0]


In [29]:
# analisis token bermakna & padding
n_bermakna = int(np.sum(token_result != 0))
n_padding  = int(np.sum(token_result == 0))

print(f"Analisis:")
print(f"Token bermakna (bukan 0) : {n_bermakna}")
print(f"Token padding  (angka 0) : {n_padding}")
print(f"Kalimat ini menggunakan {n_bermakna}/{MAX_SEQ_LEN} slot")

Analisis:
Token bermakna (bukan 0) : 22
Token padding  (angka 0) : 8
Kalimat ini menggunakan 22/30 slot


In [30]:
# detokenisasi untuk verifikasi mengubah integer menjadi kata
print(f"Detokenisasi (mengubah integer menjadi kata):")

for i, token_id in enumerate(token_result[:n_bermakna]):
    kata = vocab[token_id] if token_id < len(vocab) else "[UNK]"
    print(f"  token[{i:02d}] = {token_id:5d} → '{kata}'")

Detokenisasi (mengubah integer menjadi kata):
  token[00] =    30 → 'minta'
  token[01] =    31 → '7'
  token[02] =     6 → 'nasi'
  token[03] =   305 → 'kari'
  token[04] =   387 → 'limasari'
  token[05] =     6 → 'nasi'
  token[06] =   141 → 'putih'
  token[07] =     4 → 'ya'
  token[08] =     3 → 'sep'
  token[09] =     8 → 'oke'
  token[10] =     2 → 'kak'
  token[11] =     6 → 'nasi'
  token[12] =   305 → 'kari'
  token[13] =   387 → 'limasari'
  token[14] =     6 → 'nasi'
  token[15] =   141 → 'putih'
  token[16] =    28 → 'harganya'
  token[17] =   667 → '22'
  token[18] =    14 → 'ribu'
  token[19] =     5 → 'totalnya'
  token[20] =     9 → 'rp'
  token[21] =  3025 → '154000'


### 4.2 Demonstrasi Encoding 3 Entitas

Setiap entitas (product, quantity, price_satuan) memiliki pendekatan encoding yang berbeda karena sifat datanya berbeda.

In [31]:
# Entitas 1: PRODUCT

# verifikasi label_encoder_product yang sudah di-fit di Tahap 3
# vocabulary_size() = jumlah produk unik + 1 slot [UNK]
NUM_PRODUCTS = label_encoder_product.vocabulary_size()

print(f"Jumlah kelas (termasuk [UNK]) : {NUM_PRODUCTS}")
print(f"Produk unik di dataset        : {NUM_PRODUCTS - 1}")
print(f"Slot [UNK] (produk baru)      : 1")
print(f"Total NUM_PRODUCTS            : {NUM_PRODUCTS}")

# Catatan: NUM_PRODUCTS = 4976 (bukan 4975)
# perbedaan dari estimasi Minggu 1 karena StringLookup
# otomatis menambahkan 1 slot [UNK] di posisi 0

Jumlah kelas (termasuk [UNK]) : 4976
Produk unik di dataset        : 4975
Slot [UNK] (produk baru)      : 1
Total NUM_PRODUCTS            : 4976


In [37]:
# demonstrasi encoding beberapa nama produk
print(f"Demonstrasi encoding nama produk:")

contoh_produk = df_synthetic["product"].unique()[:5]

label_decoder_product = tf.keras.layers.StringLookup(
    vocabulary=label_encoder_product.get_vocabulary(),
    invert=True
)

for nama in contoh_produk:
    encoded = label_encoder_product([nama]).numpy()[0]
    # de-encode untuk verifikasi (invert=True)
    decoded = label_decoder_product([encoded]).numpy()[0].decode("utf-8")
    status  = "OKE" if decoded == nama else "NO"

    print(f"'{nama}' -> {encoded:4d} -> '{decoded}' {status}")

Demonstrasi encoding nama produk:
'nasi kari limasari nasi putih' ->  280 -> 'nasi kari limasari nasi putih' OKE
'mie goreng djawa' ->  912 -> 'mie goreng djawa' OKE
'teh obenk' -> 1454 -> 'teh obenk' OKE
'strawberry smoothies with ice cream' -> 3130 -> 'strawberry smoothies with ice cream' OKE
'ayam sambal ijo' -> 1319 -> 'ayam sambal ijo' OKE


In [38]:
# Entitas 2: QUANTITY

print(f"Shape y_quantity : {y_quantity.shape}")
print(f"Dtype            : {y_quantity.dtype}")
print(f"Range nilai      : {int(y_quantity.min())} - {int(y_quantity.max())} porsi")
print(f"Rata-rata        : {y_quantity.mean():.2f} porsi")

Shape y_quantity : (10050,)
Dtype            : float32
Range nilai      : 1 - 10 porsi
Rata-rata        : 5.50 porsi


In [39]:
# distribusi nilai quantity
print(f"Distribusi nilai quantity di dataset sintetis:")
unique_qty, counts_qty = np.unique(y_quantity.astype(int), return_counts=True)

for qty, count in zip(unique_qty, counts_qty):
    bar   = " " * (count // 80)
    pct   = count / len(y_quantity) * 100

    print(f"  qty={qty:2d}: {count:4d} baris ({pct:.1f}%) {bar}")

# Catatan: tidak ada normalisasi untuk quantity
# range 1-10 sudah proporsional dengan output relu
# model akan memprediksi float, dibulatkan ke int saat inferensi

Distribusi nilai quantity di dataset sintetis:
  qty= 1: 1007 baris (10.0%)             
  qty= 2:  995 baris (9.9%)             
  qty= 3: 1016 baris (10.1%)             
  qty= 4: 1003 baris (10.0%)             
  qty= 5: 1013 baris (10.1%)             
  qty= 6:  995 baris (9.9%)             
  qty= 7: 1025 baris (10.2%)             
  qty= 8:  984 baris (9.8%)             
  qty= 9:  997 baris (9.9%)             
  qty=10: 1015 baris (10.1%)             


In [40]:
# Entitas 3: PRICE_SATUAN

mask_valid    = y_price != -1
mask_sentinel = y_price == -1

print(f"Shape y_price      : {y_price.shape}")
print(f"Baris harga valid  : {mask_valid.sum()} ({mask_valid.mean():.1%})")
print(f"Baris sentinel -1  : {mask_sentinel.sum()} ({mask_sentinel.mean():.1%})")

print(f"\nStatistik harga valid sebelum normalisasi (rupiah):")
price_valid_raw = y_price[mask_valid] * 1000  # balik ke rupiah untuk display
print(f"Minimum  : Rp {price_valid_raw.min():,.0f}")
print(f"Maksimum : Rp {price_valid_raw.max():,.0f}")
print(f"Rata-rata: Rp {price_valid_raw.mean():,.0f}")

print(f"\nStatistik harga valid setelah normalisasi (÷1000):")
print(f"Minimum  : {y_price[mask_valid].min():.1f}")
print(f"Maksimum : {y_price[mask_valid].max():.1f}")
print(f"Rata-rata: {y_price[mask_valid].mean():.1f}")

Shape y_price      : (10050,)
Baris harga valid  : 8300 (82.6%)
Baris sentinel -1  : 1750 (17.4%)

Statistik harga valid sebelum normalisasi (rupiah):
Minimum  : Rp 3,000
Maksimum : Rp 75,000
Rata-rata: Rp 39,058

Statistik harga valid setelah normalisasi (÷1000):
Minimum  : 3.0
Maksimum : 75.0
Rata-rata: 39.1


In [41]:
# verifikasi jumlah price_satuan -1 tidak berubah
assert (y_price == -1).sum() == mask_sentinel.sum(), \
    "ERROR: Jumlah price_satuan -1 berubah setelah normalisasi"
print(f"Jumlah price_satuan -1 tetap utuh setelah normalisasi")

Jumlah price_satuan -1 tetap utuh setelah normalisasi


In [42]:
# perbandingan skala setelah normalisasi
print(f"Perbandingan skala (alasan normalisasi dibutuhkan):")
print(f"quantity (tanpa normalisasi) : {y_quantity.min():.1f} - {y_quantity.max():.1f}")
print(f"price_satuan (/1000)         : {y_price[mask_valid].min():.1f} - {y_price[mask_valid].max():.1f}")


# Catatan: skala lebih proporsional, loss tidak didominasi oleh price
# De-normalisasi untuk AI-2 (Denny)
# Output model (skala / 1000) x 1000 = rupiah sesungguhnya
# Contoh: model output 10.5 x 1000 = Rp 10.500

Perbandingan skala (alasan normalisasi dibutuhkan):
quantity (tanpa normalisasi) : 1.0 - 10.0
price_satuan (/1000)         : 3.0 - 75.0


### 4.3 Verifikasi Konsistensi Semua Array

Memastikan semua array memiliki panjang yang sama dan setiap baris ke-N di X, y_product, y_quantity, y_price merujuk ke transaksi yang sama (tidak ada data yang "bergeser").

In [43]:
# cek panjang semua array konsisten
print("Panjang setiap array:")
print(f"  X              : {len(X)}")
print(f"  y_product      : {len(y_product)}")
print(f"  y_quantity     : {len(y_quantity)}")
print(f"  y_price        : {len(y_price)}")

assert len(X) == len(y_product) == len(y_quantity) == len(y_price), \
    "ERROR: Panjang array tidak konsisten!"
print(f"\nSemua array konsisten - {len(X)} baris")

# verifikasi 3 sampel acak secara end-to-end
print(f"\nVerifikasi 3 sampel acak:")
np.random.seed(42)
idx_sampel = np.random.choice(len(df_synthetic), size=3, replace=False)

for i, idx in enumerate(idx_sampel):
    teks_asli   = df_synthetic["input_text"].iloc[idx]
    produk_asli = df_synthetic["product"].iloc[idx]
    qty_asli    = df_synthetic["quantity"].iloc[idx]
    price_asli  = df_synthetic["price_satuan"].iloc[idx]

    token_arr     = X[idx]
    product_int   = y_product[idx]
    qty_float     = y_quantity[idx]
    price_norm    = y_price[idx]

    # de-encode product
    produk_decoded = label_decoder_product(
        [product_int]
    ).numpy()[0].decode("utf-8")

    # de-normalisasi price (kalikan 1000, kecuali sentinel)
    price_denorm = price_norm * 1000 if price_norm != -1 else -1

Panjang setiap array:
  X              : 10050
  y_product      : 10050
  y_quantity     : 10050
  y_price        : 10050

Semua array konsisten - 10050 baris

Verifikasi 3 sampel acak:


In [45]:
# verifikasi produk cocok
cocok = "V" if produk_decoded == produk_asli else "X"

print(f"Sampel {i+1} (baris ke-{idx}):")
print(f"Teks     : '{teks_asli[:55]}...'")
print(f"Product  : '{produk_asli}' → {product_int} → '{produk_decoded}' {cocok}")
print(f"Quantity : {qty_asli} → float: {qty_float:.1f}")

if price_asli == -1:
  print(f"Price: {price_asli} (sentinel) → normalized: {price_norm}")
else:
  print(f"Price: Rp{price_asli:,} -> /1000: {price_norm:.1f} -> x1000: Rp{price_denorm:,.0f}")

assert produk_decoded == produk_asli, f"De-encoding gagal untuk '{produk_asli}'"

print(f"\nSemua sampel terverifikasi - pipeline berjalan benar")

Sampel 3 (baris ke-9596):
Teks     : 'mau 7 chicken wings ya kak [SEP] oke kak chicken wings ...'
Product  : 'chicken wings' → 1206 → 'chicken wings' V
Quantity : 7 → float: 7.0
Price: Rp23,000 -> /1000: 23.0 -> x1000: Rp23,000

Semua sampel terverifikasi - pipeline berjalan benar


### 4.4 Simpan Komponen Pipeline ke Google Drive

Menyimpan vocabulary tokenizer dan label encoder ke Google Drive
agar bisa digunakan kembali saat inferensi tanpa fit ulang dari nol.
File-file ini dibutuhkan oleh AI-2 (Denny) untuk preprocessing
di sisi FastAPI.

In [46]:
import json, os

# buat folder pipeline di Drive
# sesuaikan base path dengan struktur Drive kamu
DRIVE_BASE  = "/content/drive/MyDrive/ChatKasir/"
PIPELINE_DIR = os.path.join(DRIVE_BASE, "pipeline/")
os.makedirs(PIPELINE_DIR, exist_ok=True)

# simpan vocabulary tokenizer sebagai JSON
vocab        = vectorizer.get_vocabulary()
vocab_dict   = {word: int(idx) for idx, word in enumerate(vocab)}
vocab_path   = os.path.join(PIPELINE_DIR, "tokenizer_vocab.json")
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f"tokenizer_vocab.json disimpan ({len(vocab_dict)} kata)")

tokenizer_vocab.json disimpan (4056 kata)


In [47]:
# simpan vocabulary label encoder produk sebagai JSON
produk_vocab      = label_encoder_product.get_vocabulary()
produk_vocab_dict = {nama: int(idx) for idx, nama in enumerate(produk_vocab)}
produk_path       = os.path.join(PIPELINE_DIR, "product_vocab.json")
with open(produk_path, "w", encoding="utf-8") as f:
    json.dump(produk_vocab_dict, f, ensure_ascii=False, indent=2)

print(f"product_vocab.json disimpan ({len(produk_vocab_dict)} produk)")

product_vocab.json disimpan (4976 produk)


In [48]:
# simpan konfigurasi pipeline sebagai "kontrak teknis" untuk AI-2 (Denny)
pipeline_config = {
    "VOCAB_SIZE":       VOCAB_SIZE,
    "MAX_SEQ_LEN":      MAX_SEQ_LEN,
    "NUM_PRODUCTS":     int(NUM_PRODUCTS),
    "PRICE_NORM_FACTOR": 1000,
    "PRICE_SENTINEL":   -1,
    "catatan_untuk_denny": {
        "tokenizer":        "Gunakan tokenizer_vocab.json untuk mereplikasi TextVectorization",
        "product":          "Gunakan product_vocab.json untuk decode integer -> nama produk",
        "price_de_norm":    "Output model (price_satuan) x 1000 = rupiah sesungguhnya",
        "price_sentinel":   "Output -1 berarti model tidak menemukan harga di teks input"
    }
}

config_path = os.path.join(PIPELINE_DIR, "pipeline_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_config, f, ensure_ascii=False, indent=2)

print(f"pipeline_config.json disimpan")
print(f"\nIsi pipeline_config.json:")

print(json.dumps(pipeline_config, ensure_ascii=False, indent=2))

pipeline_config.json disimpan

Isi pipeline_config.json:
{
  "VOCAB_SIZE": 10000,
  "MAX_SEQ_LEN": 30,
  "NUM_PRODUCTS": 4976,
  "PRICE_NORM_FACTOR": 1000,
  "PRICE_SENTINEL": -1,
  "catatan_untuk_denny": {
    "tokenizer": "Gunakan tokenizer_vocab.json untuk mereplikasi TextVectorization",
    "product": "Gunakan product_vocab.json untuk decode integer -> nama produk",
    "price_de_norm": "Output model (price_satuan) x 1000 = rupiah sesungguhnya",
    "price_sentinel": "Output -1 berarti model tidak menemukan harga di teks input"
  }
}


## Ringkasan Tugas 28 April - Tokenisasi, Padding & Encoding Label

### Yang Sudah Selesai
| Entitas | Metode | Hasil |
|---------|--------|-------|
| `input_text` | TextVectorization + padding | Array (10050, 30) integer |
| `product` | StringLookup float32 | Array (10050,), NUM_PRODUCTS = 4976 |
| `quantity` | Cast float32 langsung | Array (10050,), range 1-10 |
| `price_satuan` | Normalisasi selektif /1000 | Array (10050,), sentinel -1 terjaga |

### Keputusan Teknis Penting
- **NUM_PRODUCTS = 4976** (bukan 4975): StringLookup otomatis menambah
  1 slot `[UNK]` nilai ini yang digunakan di `build_model_v1()`
- **Sentinel -1 tidak dinormalisasi**: `np.where()` menjaga -1 tetap -1
  agar `MaskedPriceLoss` bisa mendeteksi dan mengabaikannya saat training
- **De-normalisasi price untuk Denny**: output model x 1000 = rupiah

### File yang Tersimpan di Google Drive (`/pipeline/`)
- `tokenizer_vocab.json` kamus kata tokenizer
- `product_vocab.json` kamus produk label encoder
- `pipeline_config.json` konfigurasi lengkap untuk AI-2

### Target 29 April
Membangun arsitektur model v1 menggunakan TensorFlow Functional API:
Embedding -> Bidirectional LSTM -> Dense multi-output (product, quantity,
price_satuan) dan menjalankan verifikasi forward pass pertama.